# 02 · Indexing: BM25 baseline, surf-jargon synonyms, optional dense index

**Purpose.** Turn each spot profile into a searchable document and build the indexes:
1. **BM25 index:** used by the baseline *and* the improved system (the improved system adds query understanding, expansion and condition fit on top of it).
2. **Compiled synonym list:** surf jargon → corpus vocabulary, used for query expansion.
3. **Dense index (optional):** English sentence embeddings for a hybrid variant.

**Inputs:** `data/processed/spots_clean.json` (from 01), `data/resources/surf_synonyms.csv`

**Outputs:** `indexes/bm25.pkl`, `indexes/synonyms.json`, optionally `indexes/dense_embeddings.npy` + `indexes/dense_ids.json`, and `results/indexing_stats.json` (time and size, for the efficiency section)

**Why BM25 as the baseline?** It is the standard lexical ranking function (Robertson & Zaragoza, 2009). It is strong on short keyword queries and transparent, which makes it the right yardstick for showing what *condition awareness* adds.

In [ ]:
# ---- Configuration ----
import os
from pathlib import Path


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "requirements.txt").is_file():
            return folder
    raise FileNotFoundError("Project root not found. Open the project folder in VS Code (File > Open Folder).")


ROOT = find_project_root()
DATA_DIR = ROOT / "data"
CORPUS_FILE = DATA_DIR / "processed" / "spots_clean.json"
SYNONYMS_FILE = DATA_DIR / "resources" / "surf_synonyms.csv"
INDEX_DIR = ROOT / "indexes"
BM25_FILE = INDEX_DIR / "bm25.pkl"
SYNONYMS_COMPILED_FILE = INDEX_DIR / "synonyms.json"
DENSE_EMB_FILE = INDEX_DIR / "dense_embeddings.npy"
DENSE_IDS_FILE = INDEX_DIR / "dense_ids.json"
RESULTS_DIR = ROOT / "results"
INDEX_STATS_FILE = RESULTS_DIR / "indexing_stats.json"

LIBRARY_MODE = os.environ.get("SURF_IR_LIBRARY_MODE", "0") == "1"
SAMPLE_N = None            # e.g. 10 = index only the first 10 spots (development only!)
SEED = 42

BM25_K1 = 1.2              # term-frequency saturation (common default range 1.2-2.0)
BM25_B = 0.75              # document-length normalisation
USE_DENSE = False          # True requires: pip install -r requirements-dense.txt
REBUILD_DENSE = False      # False = reuse saved embeddings if they exist (encoding is the slow step)
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
DENSE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "  # recommended by the bge model card

## 1. Text analysis: tokenise → remove stopwords → stem
The same `analyze()` function processes documents and queries; using two different analysers is a classic source of silent retrieval bugs.

**Why not a standard stopword list as-is?** scikit-learn's English list contains words that carry meaning in our domain: *east, west, bottom, back, front, top, full, down*. "East coast" and "sand bottom" must stay searchable, so we keep those words.

In [ ]:
import json
import math
import pickle
import re
import time

import numpy as np
import pandas as pd
from IPython.display import display
from nltk.stem.snowball import SnowballStemmer
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

DOMAIN_KEEP = {"east", "west", "north", "south", "bottom", "back", "front", "top", "full", "down", "up",
               "off", "over", "empty", "fire", "less", "few", "first", "side", "well"}
STOPWORDS = frozenset(ENGLISH_STOP_WORDS) - DOMAIN_KEEP
_stemmer = SnowballStemmer("english")
_TOKEN_RE = re.compile(r"[a-z0-9]+")


def normalize_text(text):
    """Lower-case and turn hyphens/apostrophes into spaces ('north-east' -> 'north east', "wai'ao" -> 'wai ao')."""
    return re.sub(r"[-'’_/]", " ", str(text).lower())


def tokenize(text):
    return _TOKEN_RE.findall(normalize_text(text))


def analyze(text):
    """Tokens -> without stopwords -> Snowball stems."""
    return [_stemmer.stem(t) for t in tokenize(text) if t not in STOPWORDS]


if not LIBRARY_MODE:
    print(analyze("Uncrowded reef-break on the East coast, sandy bottom; beginners welcome!"))

## 2. From profile to document
A spot document concatenates the structured fields (rendered as words) with the free-text description. Two deliberate choices:
- **Crowd level becomes words** ("uncrowded quiet" / "crowded busy"), so an attribute query such as *uncrowded* can match even if the description never says it.
- **Orientation is *not* rendered as text.** "Faces east" would make a north-coast beach match the query *east coast*. Orientation is used by the condition-fit score instead.

In [ ]:
CROWD_WORDS = {"low": "uncrowded quiet", "medium": "", "high": "crowded busy", "unknown": ""}


def build_doc_text(spot):
    parts = [
        spot["name"], " ".join(spot.get("alt_names") or []),
        f"{spot['region']} coast", f"near {spot.get('nearest_town') or ''}",
        f"{spot['break_type']} break", f"{spot['bottom_type']} bottom",
        "for " + " ".join(spot.get("skill_levels") or []),
        CROWD_WORDS.get(spot.get("crowd_level") or "unknown", ""),
        "hazards: " + ", ".join(spot.get("hazards") or []),
        "facilities: " + ", ".join(spot.get("facilities") or []),
        spot.get("description") or "",
    ]
    return ". ".join(p for p in parts if p and p.strip())


def load_corpus_records(path=CORPUS_FILE, sample_n=SAMPLE_N):
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run 01_spot_corpus.ipynb first.")
    records = json.loads(path.read_text(encoding="utf-8"))
    return records[:sample_n] if sample_n else records


if not LIBRARY_MODE:
    corpus = load_corpus_records()
    docs = {s["spot_id"]: build_doc_text(s) for s in corpus}
    print(f"{len(docs)} documents. Example:\n")
    print(next(iter(docs.values())))

## 3. Build and save the BM25 index
`rank_bm25` keeps everything in memory, which is fine for a corpus of a few dozen documents. We pickle the index together with the document IDs and the analyser settings, so a later notebook cannot accidentally use a different analyser.

**Why we override the IDF.** `rank_bm25`'s Okapi variant uses `log((N − n + 0.5) / (n + 0.5))`, which becomes **negative** for terms that occur in more than half of the documents. In a small, homogeneous corpus that is exactly the case for words like *break* or *beach*. A document would then be *penalised* for matching the query. We use Lucene's IDF `log(1 + (N − n + 0.5) / (n + 0.5))` instead, which is always positive and is also the BM25 variant used by Lucene/Anserini/Pyserini.

In [ ]:
class BM25Lucene(BM25Okapi):
    """BM25 with Lucene's non-negative IDF."""

    def _calc_idf(self, nd):
        self.idf = {word: math.log(1 + (self.corpus_size - freq + 0.5) / (freq + 0.5)) for word, freq in nd.items()}


def build_bm25(docs):
    doc_ids = list(docs)
    doc_tokens = [analyze(docs[d]) for d in doc_ids]
    return {"bm25": BM25Lucene(doc_tokens, k1=BM25_K1, b=BM25_B), "doc_ids": doc_ids, "doc_tokens": doc_tokens,
            "doc_text": docs, "params": {"k1": BM25_K1, "b": BM25_B, "idf": "lucene", "stemmer": "snowball-english",
                                          "stopwords": "sklearn-english minus DOMAIN_KEEP"}}


def load_bm25_index(path=BM25_FILE):
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run 02_indexing.ipynb first.")
    with open(path, "rb") as f:
        return pickle.load(f)


if not LIBRARY_MODE:
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    bm25_index = build_bm25(docs)
    with open(BM25_FILE, "wb") as f:
        pickle.dump(bm25_index, f)
    bm25_seconds = time.perf_counter() - t0
    vocab = {t for toks in bm25_index["doc_tokens"] for t in toks}
    print(f"BM25: {len(docs)} docs, {len(vocab)} unique stems, built in {bm25_seconds * 1000:.1f} ms, "
          f"{BM25_FILE.stat().st_size / 1024:.1f} KB on disk")

## 4. Surf-jargon synonym list
Queries use jargon that profiles rarely contain ("beachie", "glassy", "newbie"). The team-maintained CSV maps a query term (word or phrase) to expansion terms with a weight below 1, so expansions help recall without outweighing the user's own words.

Here we validate the CSV and pre-compute the stems of both sides. Matching on stems means *beachies* and *beachie* behave the same way.

**[TEAM]** The shipped list is a *starter list*. Review every row, add terms from your query log and cite a glossary in the `source` column.

In [ ]:
def compile_synonyms(path=SYNONYMS_FILE):
    syn = pd.read_csv(path, dtype=str).fillna("")
    syn["weight"] = pd.to_numeric(syn["weight"], errors="coerce").fillna(0.5)
    bad = syn[(syn["term"] == "") | (syn["expansion"] == "")]
    if len(bad):
        raise ValueError(f"Empty term/expansion in rows {list(bad.index)} of {path.name}")
    compiled = {}
    for r in syn.itertuples():
        key = " ".join(tokenize(r.term))                      # phrase matched on normalised query text
        compiled.setdefault(key, []).append({"expansion": r.expansion, "stems": analyze(r.expansion),
                                             "weight": float(r.weight)})
    return compiled


def load_synonyms(path=SYNONYMS_COMPILED_FILE):
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run 02_indexing.ipynb first.")
    return json.loads(path.read_text(encoding="utf-8"))


if not LIBRARY_MODE:
    synonyms = compile_synonyms()
    SYNONYMS_COMPILED_FILE.write_text(json.dumps(synonyms, indent=1), encoding="utf-8")
    covered = sum(any(st in vocab for st in e["stems"]) for exps in synonyms.values() for e in exps)
    total = sum(len(exps) for exps in synonyms.values())
    print(f"{len(synonyms)} query terms, {total} expansions; {covered}/{total} expansions hit the corpus vocabulary")
    display(pd.read_csv(SYNONYMS_FILE).head(10))

## 5. Optional dense index (hybrid variant)
Lexical matching misses paraphrases ("mellow peelers" vs "gentle waves"). A small English embedding model (`bge-small-en-v1.5`, 33M parameters, runs on a laptop CPU) can catch these. It is an *optional* system variant: the core comparison of the project is BM25 vs condition-aware ranking, and the dense model adds a large install. Encoding is skipped when saved embeddings exist (`REBUILD_DENSE = False`).

In [ ]:
_dense_model = None


def get_dense_model():
    global _dense_model
    if _dense_model is None:
        from sentence_transformers import SentenceTransformer   # imported lazily: optional dependency
        _dense_model = SentenceTransformer(DENSE_MODEL, device="cpu")
    return _dense_model


def embed_query(text):
    return get_dense_model().encode([DENSE_QUERY_PREFIX + text], normalize_embeddings=True)[0]


def load_dense_index():
    """Returns (embeddings, doc_ids) or None if no dense index was built."""
    if not (DENSE_EMB_FILE.exists() and DENSE_IDS_FILE.exists()):
        return None
    return np.load(DENSE_EMB_FILE), json.loads(DENSE_IDS_FILE.read_text(encoding="utf-8"))


dense_seconds = None
if not LIBRARY_MODE and USE_DENSE:
    if REBUILD_DENSE or not DENSE_EMB_FILE.exists():
        t0 = time.perf_counter()
        emb = get_dense_model().encode(list(docs.values()), normalize_embeddings=True, show_progress_bar=True)
        np.save(DENSE_EMB_FILE, emb)
        DENSE_IDS_FILE.write_text(json.dumps(list(docs)), encoding="utf-8")
        dense_seconds = time.perf_counter() - t0
        print(f"Dense index: {emb.shape} in {dense_seconds:.1f} s")
    else:
        print("Dense index exists; skipped (set REBUILD_DENSE = True to rebuild).")
elif not LIBRARY_MODE:
    print("Dense index disabled (USE_DENSE = False).")

## 6. Indexing statistics (for the efficiency section)

In [ ]:
if not LIBRARY_MODE:
    def _kb(p):
        return round(p.stat().st_size / 1024, 1) if p.exists() else None

    index_stats = {
        "n_docs": len(docs), "vocab_size": len(vocab),
        "mean_doc_length_tokens": round(float(np.mean([len(t) for t in bm25_index["doc_tokens"]])), 1),
        "bm25_build_ms": round(bm25_seconds * 1000, 2), "bm25_size_kb": _kb(BM25_FILE),
        "dense_build_s": None if dense_seconds is None else round(dense_seconds, 2),
        "dense_size_kb": _kb(DENSE_EMB_FILE), "dense_model": DENSE_MODEL if USE_DENSE else None,
        "bm25_params": bm25_index["params"],
    }
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    INDEX_STATS_FILE.write_text(json.dumps(index_stats, indent=2), encoding="utf-8")
    display(pd.Series(index_stats, name="value").to_frame())